# EDA: WRDS RavenPack Macro Sentiment vs US Index Proxies

This notebook performs an initial exploratory analysis of whether global RavenPack macro-news sentiment lines up with short-horizon movement in major US index proxies. It uses WRDS-only data sources: RavenPack macro news and CRSP daily stock data for ETF proxies.

Target mapping: `SPX -> SPY`, `NASDAQ -> QQQ`, `Dow Jones -> DIA`, and `SOX -> SOXX`.

No raw RavenPack articles, headlines, or restricted WRDS files are saved by this notebook. All analysis is based on derived aggregates in memory.


In [ ]:
# Install dependencies in the active notebook kernel.
# Re-run only when the environment is missing packages.
%pip install -q wrds psycopg2-binary pandas numpy matplotlib seaborn pyarrow


In [ ]:
import warnings
from datetime import date

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import wrds

warnings.filterwarnings("ignore", category=FutureWarning)

pd.options.display.max_columns = 120
pd.options.display.max_rows = 80
sns.set_theme(style="whitegrid", context="notebook")


In [ ]:
# Fixed analysis parameters.
START_DATE = "2020-01-01"
END_DATE = "2025-12-31"
YEARS = range(2020, 2026)

RELEVANCE_MIN = 90
EVENT_RELEVANCE_MIN = 90
MARKET_CLOSE_ET = "16:00:00"

ETF_TO_ASSET = {
    "SPY": "SPX_proxy",
    "QQQ": "NASDAQ_100_proxy",
    "DIA": "Dow_Jones_proxy",
    "SOXX": "SOX_proxy",
}
TARGET_TICKERS = tuple(ETF_TO_ASSET.keys())

print(f"Date range: {START_DATE} to {END_DATE}")
print(f"ETF proxy universe: {TARGET_TICKERS}")


In [ ]:
# Connect to WRDS. This may prompt for credentials if no .pgpass file is configured.
db = wrds.Connection()

rp_tables = db.list_tables(library="rpna")
expected_rp_tables = [f"rpa_djpr_global_macro_{year}" for year in YEARS]
missing_rp_tables = [table for table in expected_rp_tables if table not in rp_tables]

if missing_rp_tables:
    raise RuntimeError(f"Missing RavenPack tables: {missing_rp_tables}")

print("WRDS connection ready.")
print("RavenPack macro tables found:", expected_rp_tables)


## 1. Build the institutional source universe

The source filter follows the earlier RavenPack exploration: rank-1, non-blog sources only. The dataframe `sources_df` keeps all source metadata plus boolean filter flags, while `institutional_sources_df` is the active source universe.


In [ ]:
source_query = """
    SELECT rp_entity_id, data_type, data_value
    FROM rpna.rpa_source_list
    WHERE data_type IN ('ENTITY_NAME', 'PUBLICATION_TYPE', 'SOURCE_RANK')
"""

raw_source_attributes = db.raw_sql(source_query)

sources_wide = raw_source_attributes.pivot(
    index="rp_entity_id", columns="data_type", values="data_value"
).reset_index()
sources_wide.columns.name = None

sources_df = sources_wide.rename(columns={
    "ENTITY_NAME": "source_name",
    "PUBLICATION_TYPE": "source_type",
    "SOURCE_RANK": "source_rank",
})

sources_df["source_rank"] = pd.to_numeric(sources_df["source_rank"], errors="coerce")
sources_df["is_rank1_non_blog"] = (
    sources_df["source_rank"].eq(1)
    & sources_df["source_type"].notna()
    & sources_df["source_type"].ne("BLOG")
)

institutional_sources_df = sources_df.loc[sources_df["is_rank1_non_blog"]].copy()
institutional_sources_df["weight"] = 1.0
valid_source_ids = institutional_sources_df["rp_entity_id"].dropna().astype(str).tolist()

print(f"All RavenPack source entities: {len(sources_df):,}")
print(f"Rank-1 non-blog institutional sources: {len(institutional_sources_df):,}")
display(institutional_sources_df.head())


## 2. Infer RavenPack sentiment scale

RavenPack score scales can differ by field/product version. This cell samples filtered macro records and sets transparent positive/neutral/negative thresholds before the database-side aggregation is run.


In [ ]:
def sql_string_list(values):
    """Return a SQL-safe single-quoted literal list for simple identifier strings."""
    return ", ".join("'" + str(value).replace("'", "''") + "'" for value in values)

source_id_sql = sql_string_list(valid_source_ids)

score_sample_query = f"""
    SELECT event_sentiment_score
    FROM rpna.rpa_djpr_global_macro_{max(YEARS)}
    WHERE rpa_date_utc BETWEEN '{START_DATE}' AND '{END_DATE}'
      AND relevance >= {RELEVANCE_MIN}
      AND event_relevance >= {EVENT_RELEVANCE_MIN}
      AND rp_source_id IN ({source_id_sql})
      AND event_sentiment_score IS NOT NULL
    LIMIT 10000
"""

score_sample_df = db.raw_sql(score_sample_query)
score_sample_df["event_sentiment_score"] = pd.to_numeric(
    score_sample_df["event_sentiment_score"], errors="coerce"
)
score_sample = score_sample_df["event_sentiment_score"].dropna()

if score_sample.empty:
    raise RuntimeError("No RavenPack event_sentiment_score values found for the configured filters.")

score_min = float(score_sample.min())
score_max = float(score_sample.max())
score_mean = float(score_sample.mean())

if score_min >= -1.5 and score_max <= 1.5:
    SENTIMENT_NEGATIVE_MAX = -0.05
    SENTIMENT_POSITIVE_MIN = 0.05
    SENTIMENT_SCALE_NOTE = "approx. -1 to 1 scale"
elif score_min >= 0 and score_max <= 100:
    SENTIMENT_NEGATIVE_MAX = 50.0
    SENTIMENT_POSITIVE_MIN = 50.0
    SENTIMENT_SCALE_NOTE = "approx. 0 to 100 scale"
else:
    SENTIMENT_NEGATIVE_MAX = -1.0
    SENTIMENT_POSITIVE_MIN = 1.0
    SENTIMENT_SCALE_NOTE = "wide signed scale fallback"

print({
    "sample_rows": len(score_sample),
    "min": score_min,
    "mean": score_mean,
    "max": score_max,
    "scale_note": SENTIMENT_SCALE_NOTE,
    "negative_if_less_than": SENTIMENT_NEGATIVE_MAX,
    "positive_if_greater_than": SENTIMENT_POSITIVE_MIN,
})


## 3. Pull WRDS/CRSP market data for ETF proxies

This section tries common CRSP daily stock schemas and uses the first one that returns all four ETF proxies. The resulting `market_daily_df` has one row per asset and CRSP trading session.


In [ ]:
def fetch_market_data_from_candidate(db, candidate):
    names_table = candidate["names_table"]
    daily_table = candidate["daily_table"]
    date_col = candidate["date_col"]
    ret_col = candidate["ret_col"]
    price_col = candidate["price_col"]
    volume_col = candidate["volume_col"]
    ticker_col = candidate.get("ticker_col", "ticker")
    name_start_col = candidate.get("name_start_col", "namedt")
    name_end_col = candidate.get("name_end_col", "nameendt")
    company_expr = candidate.get("company_expr", "comnam")
    ticker_sql = sql_string_list(TARGET_TICKERS)

    query = f"""
        WITH names AS (
            SELECT DISTINCT
                permno,
                {ticker_col} AS ticker,
                {name_start_col} AS name_start,
                COALESCE({name_end_col}, DATE '9999-12-31') AS name_end,
                {company_expr} AS comnam
            FROM {names_table}
            WHERE {ticker_col} IN ({ticker_sql})
              AND {name_start_col} <= DATE '{END_DATE}'
              AND COALESCE({name_end_col}, DATE '9999-12-31') >= DATE '{START_DATE}'
        )
        SELECT
            d.{date_col} AS session_date,
            n.ticker,
            n.comnam,
            d.permno,
            d.{ret_col} AS daily_return,
            d.{price_col} AS price,
            d.{volume_col} AS volume
        FROM {daily_table} d
        INNER JOIN names n
            ON d.permno = n.permno
           AND d.{date_col} BETWEEN n.name_start AND n.name_end
        WHERE d.{date_col} BETWEEN DATE '{START_DATE}' AND DATE '{END_DATE}'
        ORDER BY n.ticker, d.{date_col}
    """
    return db.raw_sql(query)

crsp_candidates = [
    {
        "label": "CRSP CIZ crsp.dsf_v2",
        "names_table": "crsp.stksecurityinfohist",
        "daily_table": "crsp.dsf_v2",
        "date_col": "dlycaldt",
        "ret_col": "dlyret",
        "price_col": "dlyprc",
        "volume_col": "dlyvol",
        "ticker_col": "ticker",
        "name_start_col": "secinfostartdt",
        "name_end_col": "secinfoenddt",
        "company_expr": "NULL::text",
    },
    {
        "label": "legacy crsp.dsf",
        "names_table": "crsp.stocknames",
        "daily_table": "crsp.dsf",
        "date_col": "date",
        "ret_col": "ret",
        "price_col": "prc",
        "volume_col": "vol",
        "ticker_col": "ticker",
        "name_start_col": "namedt",
        "name_end_col": "nameendt",
        "company_expr": "comnam",
    },
    {
        "label": "crspa.dsf",
        "names_table": "crspa.stocknames",
        "daily_table": "crspa.dsf",
        "date_col": "date",
        "ret_col": "ret",
        "price_col": "prc",
        "volume_col": "vol",
        "ticker_col": "ticker",
        "name_start_col": "namedt",
        "name_end_col": "nameendt",
        "company_expr": "comnam",
    },
    {
        "label": "crsp_a_stock.dsf",
        "names_table": "crsp_a_stock.stocknames",
        "daily_table": "crsp_a_stock.dsf",
        "date_col": "date",
        "ret_col": "ret",
        "price_col": "prc",
        "volume_col": "vol",
        "ticker_col": "ticker",
        "name_start_col": "namedt",
        "name_end_col": "nameendt",
        "company_expr": "comnam",
    },
]

market_daily_raw = None
market_candidate_used = None
candidate_errors = []

for candidate in crsp_candidates:
    try:
        candidate_df = fetch_market_data_from_candidate(db, candidate)
        tickers_found = set(candidate_df["ticker"].dropna().unique()) if not candidate_df.empty else set()
        if set(TARGET_TICKERS).issubset(tickers_found):
            market_daily_raw = candidate_df.copy()
            market_candidate_used = candidate["label"]
            break
        candidate_errors.append((candidate["label"], f"found tickers {sorted(tickers_found)}"))
    except Exception as exc:
        candidate_errors.append((candidate["label"], repr(exc)))

if market_daily_raw is None:
    raise RuntimeError(
        "Could not retrieve all ETF proxies from the configured WRDS/CRSP candidates. "
        f"Candidate results: {candidate_errors}"
    )

market_daily_df = market_daily_raw.copy()
market_daily_df["session_date"] = pd.to_datetime(market_daily_df["session_date"])
market_daily_df["daily_return"] = pd.to_numeric(market_daily_df["daily_return"], errors="coerce")
market_daily_df["price"] = pd.to_numeric(market_daily_df["price"], errors="coerce").abs()
market_daily_df["volume"] = pd.to_numeric(market_daily_df["volume"], errors="coerce")
market_daily_df["asset"] = market_daily_df["ticker"].map(ETF_TO_ASSET)

market_daily_df = (
    market_daily_df
    .dropna(subset=["asset", "session_date"])
    .sort_values(["asset", "session_date", "permno"])
    .drop_duplicates(["asset", "session_date"], keep="last")
    .reset_index(drop=True)
)

for horizon in [1, 5]:
    shifted_product = pd.Series(1.0, index=market_daily_df.index)
    for lag in range(1, horizon + 1):
        shifted_product = shifted_product * (1 + market_daily_df.groupby("asset")["daily_return"].shift(-lag))
    fwd_col = f"fwd_{horizon}d_return"
    label_col = f"fwd_{horizon}d_positive"
    market_daily_df[fwd_col] = shifted_product - 1
    market_daily_df[label_col] = np.where(
        market_daily_df[fwd_col].notna(),
        (market_daily_df[fwd_col] > 0).astype(float),
        np.nan,
    )

print(f"CRSP source used: {market_candidate_used}")
print(market_daily_df.groupby(["asset", "ticker"]).agg(
    start=("session_date", "min"),
    end=("session_date", "max"),
    sessions=("session_date", "nunique"),
    nonnull_returns=("daily_return", "count"),
))
display(market_daily_df.head())



## 4. Aggregate RavenPack macro news by signal date

The SQL below converts UTC timestamps to Eastern Time, applies the 4:00 PM ET cutoff, and aggregates filtered RavenPack records before moving the result into pandas. Calendar dates that fall on weekends or holidays are mapped to the next CRSP trading session in pandas.


In [ ]:
ET_TIMESTAMP_SQL = "((timestamp_utc AT TIME ZONE 'UTC') AT TIME ZONE 'America/New_York')"


def fetch_news_daily_for_year(year):
    table_name = f"rpna.rpa_djpr_global_macro_{year}"
    query = f"""
        SELECT
            CASE
                WHEN {ET_TIMESTAMP_SQL}::time < TIME '{MARKET_CLOSE_ET}'
                    THEN {ET_TIMESTAMP_SQL}::date
                ELSE ({ET_TIMESTAMP_SQL}::date + INTERVAL '1 day')::date
            END AS signal_calendar_date,
            COUNT(*)::bigint AS event_record_count,
            COUNT(DISTINCT rp_story_id)::bigint AS unique_story_count,
            AVG(event_sentiment_score)::float AS mean_event_sentiment_score,
            SUM(CASE WHEN event_sentiment_score > {SENTIMENT_POSITIVE_MIN} THEN 1 ELSE 0 END)::bigint AS positive_event_count,
            SUM(CASE WHEN event_sentiment_score < {SENTIMENT_NEGATIVE_MAX} THEN 1 ELSE 0 END)::bigint AS negative_event_count,
            SUM(CASE WHEN event_sentiment_score >= {SENTIMENT_NEGATIVE_MAX}
                      AND event_sentiment_score <= {SENTIMENT_POSITIVE_MIN}
                     THEN 1 ELSE 0 END)::bigint AS neutral_event_count,
            COUNT(DISTINCT rp_source_id)::bigint AS unique_source_count
        FROM {table_name}
        WHERE rpa_date_utc BETWEEN DATE '{START_DATE}' AND DATE '{END_DATE}'
          AND relevance >= {RELEVANCE_MIN}
          AND event_relevance >= {EVENT_RELEVANCE_MIN}
          AND rp_source_id IN ({source_id_sql})
          AND timestamp_utc IS NOT NULL
          AND event_sentiment_score IS NOT NULL
        GROUP BY 1
        ORDER BY 1
    """
    df = db.raw_sql(query)
    df["source_year"] = year
    return df

news_daily_parts = []
for year in YEARS:
    print(f"Aggregating RavenPack macro news for {year}...")
    news_daily_parts.append(fetch_news_daily_for_year(year))

news_signal_calendar_df = pd.concat(news_daily_parts, ignore_index=True)
news_signal_calendar_df["signal_calendar_date"] = pd.to_datetime(news_signal_calendar_df["signal_calendar_date"])

count_columns = [
    "event_record_count",
    "unique_story_count",
    "positive_event_count",
    "negative_event_count",
    "neutral_event_count",
    "unique_source_count",
]
for col in count_columns:
    news_signal_calendar_df[col] = pd.to_numeric(news_signal_calendar_df[col], errors="coerce").fillna(0)
news_signal_calendar_df["mean_event_sentiment_score"] = pd.to_numeric(
    news_signal_calendar_df["mean_event_sentiment_score"], errors="coerce"
)

print(f"Calendar signal rows: {len(news_signal_calendar_df):,}")
display(news_signal_calendar_df.head())


In [ ]:
def map_to_next_trading_session(dates, trading_sessions):
    sessions = pd.Series(pd.to_datetime(trading_sessions)).drop_duplicates().sort_values()
    session_values = sessions.to_numpy(dtype="datetime64[ns]")
    target_values = pd.to_datetime(dates).to_numpy(dtype="datetime64[ns]")
    positions = np.searchsorted(session_values, target_values, side="left")
    mapped = np.full(len(target_values), np.datetime64("NaT"), dtype="datetime64[ns]")
    valid = positions < len(session_values)
    mapped[valid] = session_values[positions[valid]]
    return pd.to_datetime(mapped)

trading_sessions = market_daily_df["session_date"].drop_duplicates().sort_values()

news_aligned = news_signal_calendar_df.copy()
news_aligned["session_date"] = map_to_next_trading_session(
    news_aligned["signal_calendar_date"], trading_sessions
)
news_aligned = news_aligned.dropna(subset=["session_date"])

weighted_score_numerator = (
    news_aligned["mean_event_sentiment_score"] * news_aligned["event_record_count"]
)
news_aligned["sentiment_weighted_sum"] = weighted_score_numerator

news_daily_df = (
    news_aligned
    .groupby("session_date", as_index=False)
    .agg(
        event_record_count=("event_record_count", "sum"),
        unique_story_count=("unique_story_count", "sum"),
        positive_event_count=("positive_event_count", "sum"),
        negative_event_count=("negative_event_count", "sum"),
        neutral_event_count=("neutral_event_count", "sum"),
        unique_source_count=("unique_source_count", "max"),
        sentiment_weighted_sum=("sentiment_weighted_sum", "sum"),
    )
)

news_daily_df["mean_event_sentiment_score"] = (
    news_daily_df["sentiment_weighted_sum"] / news_daily_df["event_record_count"]
)
news_daily_df = news_daily_df.drop(columns=["sentiment_weighted_sum"])

for label in ["positive", "negative", "neutral"]:
    news_daily_df[f"{label}_event_share"] = (
        news_daily_df[f"{label}_event_count"] / news_daily_df["event_record_count"]
    )

conditions = [
    news_daily_df["mean_event_sentiment_score"] > SENTIMENT_POSITIVE_MIN,
    news_daily_df["mean_event_sentiment_score"] < SENTIMENT_NEGATIVE_MAX,
]
news_daily_df["sentiment_bucket"] = np.select(conditions, ["positive", "negative"], default="neutral")

print(f"Trading-session news rows: {len(news_daily_df):,}")
display(news_daily_df.head())


## 5. Topic and group summaries

These are separate aggregate queries for EDA plots. They are not joined into the daily market panel.


In [ ]:
def fetch_category_counts_for_year(year, category_col, output_col):
    table_name = f"rpna.rpa_djpr_global_macro_{year}"
    query = f"""
        SELECT
            EXTRACT(YEAR FROM rpa_date_utc)::int AS year,
            COALESCE({category_col}, 'Unknown') AS {output_col},
            COUNT(*)::bigint AS event_record_count,
            COUNT(DISTINCT rp_story_id)::bigint AS unique_story_count
        FROM {table_name}
        WHERE rpa_date_utc BETWEEN DATE '{START_DATE}' AND DATE '{END_DATE}'
          AND relevance >= {RELEVANCE_MIN}
          AND event_relevance >= {EVENT_RELEVANCE_MIN}
          AND rp_source_id IN ({source_id_sql})
          AND event_sentiment_score IS NOT NULL
        GROUP BY 1, 2
        ORDER BY 1, 3 DESC
    """
    return db.raw_sql(query)

topic_counts_df = pd.concat(
    [fetch_category_counts_for_year(year, "topic", "topic") for year in YEARS],
    ignore_index=True,
)
group_counts_df = pd.concat(
    [fetch_category_counts_for_year(year, '"group"', "group_name") for year in YEARS],
    ignore_index=True,
)

for df in [topic_counts_df, group_counts_df]:
    df["event_record_count"] = pd.to_numeric(df["event_record_count"], errors="coerce")
    df["unique_story_count"] = pd.to_numeric(df["unique_story_count"], errors="coerce")

print("Top topics:")
display(topic_counts_df.groupby("topic", as_index=False)["event_record_count"].sum().sort_values("event_record_count", ascending=False).head(15))
print("Top groups:")
display(group_counts_df.groupby("group_name", as_index=False)["event_record_count"].sum().sort_values("event_record_count", ascending=False).head(15))


## 6. Join news features to ETF proxy returns

Global macro-news features are common across assets for a trading session. The joined panel repeats daily news features for each asset so each row is one asset-session observation.


In [ ]:
eda_panel_df = market_daily_df.merge(news_daily_df, on="session_date", how="left")

news_fill_zero_cols = [
    "event_record_count",
    "unique_story_count",
    "positive_event_count",
    "negative_event_count",
    "neutral_event_count",
    "unique_source_count",
    "positive_event_share",
    "negative_event_share",
    "neutral_event_share",
]
for col in news_fill_zero_cols:
    eda_panel_df[col] = eda_panel_df[col].fillna(0)

eda_panel_df["has_macro_news"] = eda_panel_df["event_record_count"] > 0
eda_panel_df["year"] = eda_panel_df["session_date"].dt.year
eda_panel_df["month"] = eda_panel_df["session_date"].dt.to_period("M").dt.to_timestamp()

print(f"Panel shape: {eda_panel_df.shape}")
print("Rows by asset:")
display(eda_panel_df.groupby(["asset", "ticker"]).size().rename("rows"))
display(eda_panel_df.head())


## 7. Validation checks

These checks verify the core data shape, news aggregation, sentiment shares, and forward-return construction.


In [ ]:
expected_assets = set(ETF_TO_ASSET.values())
actual_assets = set(market_daily_df["asset"].dropna().unique())
assert expected_assets.issubset(actual_assets), f"Missing assets: {expected_assets - actual_assets}"

panel_duplicate_count = eda_panel_df.duplicated(["asset", "session_date"]).sum()
assert panel_duplicate_count == 0, f"Duplicate asset-session rows found: {panel_duplicate_count}"

if not news_daily_df.empty:
    assert (news_daily_df["unique_story_count"] <= news_daily_df["event_record_count"]).all(), (
        "unique_story_count should not exceed event_record_count"
    )
    share_sum = news_daily_df[["positive_event_share", "negative_event_share", "neutral_event_share"]].sum(axis=1)
    assert np.allclose(share_sum, 1.0, atol=1e-6), "Sentiment shares do not sum to 1.0"

# Forward return spot check: today's fwd_1d_return must equal tomorrow's daily_return by asset.
check_df = market_daily_df.sort_values(["asset", "session_date"]).copy()
check_df["next_daily_return"] = check_df.groupby("asset")["daily_return"].shift(-1)
valid_check = check_df[["fwd_1d_return", "next_daily_return"]].dropna()
assert np.allclose(valid_check["fwd_1d_return"], valid_check["next_daily_return"], atol=1e-12), (
    "fwd_1d_return is not shifted correctly"
)

coverage_summary = eda_panel_df.groupby("asset").agg(
    sessions=("session_date", "nunique"),
    sessions_with_macro_news=("has_macro_news", "sum"),
    missing_fwd_1d=("fwd_1d_return", lambda s: s.isna().sum()),
    missing_fwd_5d=("fwd_5d_return", lambda s: s.isna().sum()),
)
coverage_summary["macro_news_coverage_share"] = (
    coverage_summary["sessions_with_macro_news"] / coverage_summary["sessions"]
)

display(coverage_summary)
print("Validation checks passed.")


In [ ]:
# Spot-check after-hours alignment without retrieving article text.
after_hours_query = f"""
    SELECT
        timestamp_utc,
        {ET_TIMESTAMP_SQL} AS timestamp_et,
        CASE
            WHEN {ET_TIMESTAMP_SQL}::time < TIME '{MARKET_CLOSE_ET}'
                THEN {ET_TIMESTAMP_SQL}::date
            ELSE ({ET_TIMESTAMP_SQL}::date + INTERVAL '1 day')::date
        END AS signal_calendar_date,
        event_sentiment_score,
        source_name
    FROM rpna.rpa_djpr_global_macro_{max(YEARS)}
    WHERE rpa_date_utc BETWEEN DATE '{START_DATE}' AND DATE '{END_DATE}'
      AND relevance >= {RELEVANCE_MIN}
      AND event_relevance >= {EVENT_RELEVANCE_MIN}
      AND rp_source_id IN ({source_id_sql})
      AND timestamp_utc IS NOT NULL
      AND {ET_TIMESTAMP_SQL}::time >= TIME '{MARKET_CLOSE_ET}'
    LIMIT 10
"""

after_hours_spot_check_df = db.raw_sql(after_hours_query)
after_hours_spot_check_df["signal_calendar_date"] = pd.to_datetime(after_hours_spot_check_df["signal_calendar_date"])
after_hours_spot_check_df["mapped_session_date"] = map_to_next_trading_session(
    after_hours_spot_check_df["signal_calendar_date"], trading_sessions
)
display(after_hours_spot_check_df)


## 8. EDA plots


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4))

news_yearly = news_daily_df.assign(year=news_daily_df["session_date"].dt.year).groupby("year", as_index=False)["event_record_count"].sum()
sns.barplot(data=news_yearly, x="year", y="event_record_count", ax=axes[0], color="#4C78A8")
axes[0].set_title("Yearly RavenPack macro event volume")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Event records")

news_monthly = news_daily_df.assign(month=news_daily_df["session_date"].dt.to_period("M").dt.to_timestamp()).groupby("month", as_index=False)["event_record_count"].sum()
sns.lineplot(data=news_monthly, x="month", y="event_record_count", ax=axes[1], color="#F58518")
axes[1].set_title("Monthly RavenPack macro event volume")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Event records")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4))

sns.histplot(score_sample, bins=40, kde=True, ax=axes[0], color="#54A24B")
axes[0].axvline(SENTIMENT_NEGATIVE_MAX, color="red", linestyle="--", linewidth=1)
axes[0].axvline(SENTIMENT_POSITIVE_MIN, color="green", linestyle="--", linewidth=1)
axes[0].set_title("Sampled event sentiment score distribution")
axes[0].set_xlabel("Event sentiment score")

trend_df = news_daily_df.sort_values("session_date").copy()
trend_df["sentiment_20d_avg"] = trend_df["mean_event_sentiment_score"].rolling(20, min_periods=5).mean()
sns.lineplot(data=trend_df, x="session_date", y="sentiment_20d_avg", ax=axes[1], color="#B279A2")
axes[1].set_title("20-session rolling mean macro sentiment")
axes[1].set_xlabel("Session date")
axes[1].set_ylabel("Mean event sentiment score")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

top_topics = (
    topic_counts_df.groupby("topic", as_index=False)["event_record_count"].sum()
    .sort_values("event_record_count", ascending=False)
    .head(12)
)
sns.barplot(data=top_topics, y="topic", x="event_record_count", ax=axes[0], color="#4C78A8")
axes[0].set_title("Top RavenPack macro topics")
axes[0].set_xlabel("Event records")
axes[0].set_ylabel("")

top_groups = (
    group_counts_df.groupby("group_name", as_index=False)["event_record_count"].sum()
    .sort_values("event_record_count", ascending=False)
    .head(12)
)
sns.barplot(data=top_groups, y="group_name", x="event_record_count", ax=axes[1], color="#F58518")
axes[1].set_title("Top RavenPack macro groups")
axes[1].set_xlabel("Event records")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4))

sns.histplot(
    data=market_daily_df,
    x="daily_return",
    hue="asset",
    bins=80,
    element="step",
    stat="density",
    common_norm=False,
    ax=axes[0],
)
axes[0].set_title("Daily return distribution by ETF proxy")
axes[0].set_xlabel("Daily return")

sns.boxplot(data=market_daily_df, x="asset", y="daily_return", ax=axes[1])
axes[1].set_title("Daily returns by ETF proxy")
axes[1].set_xlabel("")
axes[1].set_ylabel("Daily return")
axes[1].tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.show()


In [ ]:
coverage_plot_df = coverage_summary.reset_index()

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
sns.barplot(data=coverage_plot_df, x="asset", y="sessions", ax=axes[0], color="#4C78A8")
axes[0].set_title("CRSP sessions by asset")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=25)

sns.barplot(data=coverage_plot_df, x="asset", y="macro_news_coverage_share", ax=axes[1], color="#54A24B")
axes[1].set_title("Share of sessions with macro news")
axes[1].set_xlabel("")
axes[1].set_ylabel("Coverage share")
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.show()


In [ ]:
correlation_columns = [
    "event_record_count",
    "unique_story_count",
    "mean_event_sentiment_score",
    "positive_event_share",
    "negative_event_share",
    "neutral_event_share",
    "daily_return",
    "fwd_1d_return",
    "fwd_5d_return",
    "volume",
]

corr_df = eda_panel_df[correlation_columns].replace([np.inf, -np.inf], np.nan).dropna(how="all")
correlation_matrix = corr_df.corr(numeric_only=True)

plt.figure(figsize=(10, 7))
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="vlag", center=0, linewidths=0.5)
plt.title("Correlation heatmap: macro sentiment features and ETF returns")
plt.tight_layout()
plt.show()


In [ ]:
bucket_return_df = (
    eda_panel_df.dropna(subset=["sentiment_bucket", "fwd_1d_return", "fwd_5d_return"])
    .groupby(["asset", "sentiment_bucket"], as_index=False)
    .agg(
        observations=("session_date", "count"),
        avg_fwd_1d_return=("fwd_1d_return", "mean"),
        avg_fwd_5d_return=("fwd_5d_return", "mean"),
        positive_1d_rate=("fwd_1d_positive", "mean"),
        positive_5d_rate=("fwd_5d_positive", "mean"),
    )
)

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
sns.barplot(data=bucket_return_df, x="asset", y="avg_fwd_1d_return", hue="sentiment_bucket", ax=axes[0])
axes[0].set_title("Average next-day return by macro sentiment bucket")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=25)

sns.barplot(data=bucket_return_df, x="asset", y="avg_fwd_5d_return", hue="sentiment_bucket", ax=axes[1])
axes[1].set_title("Average next-5-session return by macro sentiment bucket")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.show()

display(bucket_return_df)


In [ ]:
volume_panel_df = eda_panel_df.copy()
volume_panel_df["news_volume_group"] = "no_news"
positive_volume = volume_panel_df["event_record_count"] > 0
volume_cutoff = volume_panel_df.loc[positive_volume, "event_record_count"].median()
volume_panel_df.loc[positive_volume & (volume_panel_df["event_record_count"] <= volume_cutoff), "news_volume_group"] = "low_news_volume"
volume_panel_df.loc[positive_volume & (volume_panel_df["event_record_count"] > volume_cutoff), "news_volume_group"] = "high_news_volume"

volume_return_df = (
    volume_panel_df.dropna(subset=["fwd_1d_return", "fwd_5d_return"])
    .groupby(["asset", "news_volume_group"], as_index=False)
    .agg(
        observations=("session_date", "count"),
        avg_fwd_1d_return=("fwd_1d_return", "mean"),
        avg_fwd_5d_return=("fwd_5d_return", "mean"),
        positive_1d_rate=("fwd_1d_positive", "mean"),
    )
)

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
sns.barplot(data=volume_return_df, x="asset", y="avg_fwd_1d_return", hue="news_volume_group", ax=axes[0])
axes[0].set_title("Average next-day return by news volume group")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=25)

sns.barplot(data=volume_return_df, x="asset", y="avg_fwd_5d_return", hue="news_volume_group", ax=axes[1])
axes[1].set_title("Average next-5-session return by news volume group")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.show()

display(volume_return_df)


## 9. Interpretation checklist

Use this final checklist when writing the proposal/report section:

- Did macro-news volume cluster around known market stress periods?
- Is macro-news sentiment mostly neutral, or does it have enough variation for modeling?
- Do high-volume macro-news days behave differently from low-volume days?
- Do positive, neutral, and negative sentiment buckets show meaningfully different forward returns?
- Are any relationships consistent across all four ETF proxies, or concentrated in one proxy such as `SOXX`?
- Are the observed effects large enough to justify a later predictive-modeling notebook?

This notebook is descriptive. Treat any apparent relationship as hypothesis generation until evaluated with a leakage-controlled train/test design.
